# Embedding models: from word vectors to sentences

A cosmetics brand lets shoppers search its reviews in Korean or English, and the search runs on an English-only
embedding model. In this notebook you run that real model with sentence-transformers on the lesson's 24 reviews,
see how it cuts Korean into single letters, count its hits, do mean pooling and cosine by hand, and compare it
with the multilingual model's recorded results.

About 5 minutes. Run the cells from top to bottom. The model download is about 90 MB. No account or key is needed.

## 1. Install the library

The next cell installs sentence-transformers, the version the lesson was checked with. It may print a few lines
from pip. If Colab asks you to restart the session, do it and run the cells again from the top.

In [1]:
%pip install -q sentence-transformers==6.3.0

Note: you may need to restart the kernel to use updated packages.


## 2. The reviews and the queries

The same 24 reviews (12 in English, 12 in Korean) and 10 shopper queries as the lesson. They were written for the
lesson, not by real customers. Each review has a topic. A query's right answers are the two reviews with the same
topic: one in English and one in Korean. `hits` counts how many right answers are in a model's top 3.

**Look for:** 24 reviews, 10 queries, 20 possible hits.

In [2]:
REVIEWS = [
    ("r01", "en", "clear",   "This sunscreen goes on clear and leaves no chalky film on my face."),
    ("r02", "ko", "clear",   "백탁 현상이 전혀 없고 투명하게 스며들어요."),
    ("r03", "en", "sting",   "The sunscreen ran into my eyes when I sweated and they stung for an hour."),
    ("r04", "ko", "sting",   "땀 흘리니까 선크림이 눈에 들어가서 너무 따가웠어요."),
    ("r05", "en", "dry",     "My dry skin finally feels soft and hydrated all day with this cream."),
    ("r06", "ko", "dry",     "건조한 피부인데 이 크림 바르고 하루 종일 촉촉해요."),
    ("r07", "en", "greasy",  "Way too heavy and greasy for oily skin, my face shines by noon."),
    ("r08", "ko", "greasy",  "지성 피부에는 너무 무겁고 번들거려요. 점심쯤이면 얼굴이 기름져요."),
    ("r09", "en", "gentle",  "A gentle cleanser that doesn't leave my face tight after washing."),
    ("r10", "ko", "gentle",  "세안 후에도 당기지 않고 순한 클렌저예요."),
    ("r11", "en", "scent",   "The cleanser smells like strong perfume, the scent gave me a headache."),
    ("r12", "ko", "scent",   "클렌저 향이 너무 강해서 머리가 아팠어요."),
    ("r13", "en", "lasting", "The lip tint stayed on through lunch and coffee without fading."),
    ("r14", "ko", "lasting", "립틴트가 밥 먹고 커피 마셔도 지워지지 않고 오래가요."),
    ("r15", "en", "flaky",   "This tint dries out my lips and makes them flaky."),
    ("r16", "ko", "flaky",   "틴트 바르면 입술이 건조해지고 각질이 일어나요."),
    ("r17", "en", "acne",    "After two weeks of this toner my breakouts calmed down."),
    ("r18", "ko", "acne",    "이 토너 2주 쓰고 나서 여드름이 많이 가라앉았어요."),
    ("r19", "en", "itchy",   "The toner made my cheeks sting and itch, so I stopped using it."),
    ("r20", "ko", "itchy",   "토너 쓰고 볼이 화끈거리고 가려워서 사용을 중단했어요."),
    ("r21", "en", "leak",    "The bottle arrived cracked and half the serum had leaked into the box."),
    ("r22", "ko", "leak",    "배송 왔는데 병이 깨져서 세럼이 상자에 다 새어 있었어요."),
    ("r23", "en", "oxidize", "The cushion shade matched at first but oxidized after an hour."),
    ("r24", "ko", "oxidize", "쿠션이 처음엔 잘 맞는데 한 시간 지나면 산화돼서 톤이 안 맞아요."),
]
QUERIES = [
    ("q01", "ko", "clear",   "백탁 없는 선크림"),            # sunscreen without a chalky cast
    ("q02", "ko", "greasy",  "지성 피부에 번들거리는 크림"),  # cream that turns shiny on oily skin
    ("q03", "ko", "flaky",   "입술이 건조해지는 틴트"),       # tint that dries the lips
    ("q04", "ko", "scent",   "향이 강한 클렌저"),            # cleanser with a strong scent
    ("q05", "ko", "leak",    "배송 중 파손"),                # damaged in shipping
    ("q06", "en", "sting",   "sunscreen that stings the eyes"),
    ("q07", "en", "dry",     "moisturizer for dry skin"),
    ("q08", "en", "lasting", "long-lasting lip tint"),
    ("q09", "en", "acne",    "toner that cleared my acne"),
    ("q10", "en", "oxidize", "foundation that oxidizes"),
]
reviews = [text for _, _, _, text in REVIEWS]
review_ids = [rid for rid, _, _, _ in REVIEWS]
lang = {rid: lg for rid, lg, _, _ in REVIEWS} | {qid: lg for qid, lg, _, _ in QUERIES}
topic = {rid: tp for rid, _, tp, _ in REVIEWS} | {qid: tp for qid, _, tp, _ in QUERIES}

def hits(top3, query_lang=None, other_language_only=False):
    """top3 = {query id: [(review id, score), ...]}. Counts right answers in the top 3."""
    n = 0
    for qid, found in top3.items():
        if query_lang and lang[qid] != query_lang:
            continue
        for rid, _ in found:
            if topic[rid] == topic[qid] and (not other_language_only or lang[rid] != lang[qid]):
                n += 1
    return n

print("reviews:", len(REVIEWS), " queries:", len(QUERIES), " possible hits:", 2 * len(QUERIES))

reviews: 24  queries: 10  possible hits: 20


## 3. Load the English model

`all-MiniLM-L6-v2` is the English model the shop uses now. `revision` pins the exact model files the lesson used,
so your numbers match the lesson's. The cell prints `max_seq_length` (texts longer than this many tokens are cut
without warning) and the vector size.

**Look for:** `max_seq_length: 256` and `vector size: 384`.

In [3]:
import os, warnings
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
warnings.filterwarnings("ignore")
import numpy as np
import torch
import transformers
from sentence_transformers import SentenceTransformer

transformers.logging.set_verbosity_error()
torch.manual_seed(0)  # nothing here is random (the model only reads), but we seed anyway

name = "sentence-transformers/all-MiniLM-L6-v2"
model = SentenceTransformer(name, revision="1110a243fdf4706b3f48f1d95db1a4f5529b4d41", device="cpu")
print("max_seq_length:", model.max_seq_length)
print("vector size:", model.get_sentence_embedding_dimension())

max_seq_length: 256
vector size: 384


## 4. What the tokenizer sees

Before the model makes any numbers, its tokenizer cuts the text into pieces from a fixed vocabulary.
The cell cuts the Korean query q01 and the English query q06.

**Look for:** 17 pieces for the Korean query: single Korean letters, one `[UNK]` (a piece the vocabulary does not
have), plus the markers `[CLS]` and `[SEP]`. The English query comes out as real word pieces.

In [4]:
tok = model.tokenizer
print("vocabulary size:", tok.vocab_size)
for text in ["백탁 없는 선크림", "sunscreen that stings the eyes"]:
    pieces = tok.convert_ids_to_tokens(tok(text)["input_ids"])
    print(len(pieces), "pieces:", pieces)

vocabulary size: 30522
17 pieces: ['[CLS]', 'ᄇ', '##ᅢ', '##ᆨ', '##ᄐ', '##ᅡ', '##ᆨ', '[UNK]', 'ᄉ', '##ᅥ', '##ᆫ', '##ᄏ', '##ᅳ', '##ᄅ', '##ᅵ', '##ᆷ', '[SEP]']
9 pieces: ['[CLS]', 'suns', '##creen', 'that', 'sting', '##s', 'the', 'eyes', '[SEP]']


## 5. Turn every review into a vector

`encode_document` runs the tokenizer, the model and mean pooling on each review. `normalize_embeddings=True` scales
every vector to length 1, so a dot product alone is the cosine.

**Look for:** shape `(24, 384)`: one vector of 384 numbers per review, however long the review. The length of a
vector is 1.0.

In [5]:
review_vecs = model.encode_document(reviews, batch_size=32, normalize_embeddings=True)
print("shape:", review_vecs.shape)
print("length of the first vector:", round(float(np.linalg.norm(review_vecs[0])), 4))
print("first 3 numbers of r01:", review_vecs[0][:3].round(4))

shape: (24, 384)
length of the first vector: 1.0
first 3 numbers of r01: [ 0.0336 -0.0178  0.0301]


## 6. Search with the Korean query

This is the code from the lesson's library screen. `model.similarity` computes the cosine between the query vector
and every review vector, and `topk(3)` keeps the three highest scores.

**Look for:** `['r04', 'r12', 'r20']` with scores `0.8635, 0.7196, 0.6608`. All three are wrong: the query asks for
a sunscreen without a chalky cast (r01, r02), and these are about stinging eyes, a strong scent and itchy cheeks.

In [6]:
query_vecs = model.encode_query(["백탁 없는 선크림"], normalize_embeddings=True)
scores = model.similarity(query_vecs, review_vecs)
top = scores[0].topk(3)
print([review_ids[i] for i in top.indices], top.values.round(decimals=4))

['r04', 'r12', 'r20'] tensor([0.8635, 0.7196, 0.6608])


## 7. All 10 queries, counted

The cell runs every query, prints its top 3 (a `*` marks a right answer) and counts hits the way you did by hand in
the lesson.

**Look for:** 3 hits on Korean queries, 5 on English queries, 8 of 20 in all, and 0 hits in the other language.
The English model never finds a review across languages.

In [7]:
query_ids = [qid for qid, _, _, _ in QUERIES]
all_query_vecs = model.encode_query([q for _, _, _, q in QUERIES], normalize_embeddings=True)
S = model.similarity(all_query_vecs, review_vecs)
TOP3_EN = {}
for qi, qid in enumerate(query_ids):
    t = S[qi].topk(3)
    TOP3_EN[qid] = [(review_ids[i], round(float(v), 4)) for v, i in zip(t.values, t.indices)]
    marked = [f"{rid}{'*' if topic[rid] == topic[qid] else ''} {sc:.4f}" for rid, sc in TOP3_EN[qid]]
    print(qid, QUERIES[qi][3], "->", ", ".join(marked))
print()
print("hits, Korean queries: ", hits(TOP3_EN, "ko"), "of 10")
print("hits, English queries:", hits(TOP3_EN, "en"), "of 10")
print("hits in the other language:", hits(TOP3_EN, other_language_only=True))
print("all hits:", hits(TOP3_EN), "of 20")

q01 백탁 없는 선크림 -> r04 0.8635, r12 0.7196, r20 0.6608
q02 지성 피부에 번들거리는 크림 -> r12 0.8398, r06 0.8016, r08* 0.7648
q03 입술이 건조해지는 틴트 -> r14 0.8355, r08 0.8019, r16* 0.7742
q04 향이 강한 클렌저 -> r12* 0.8611, r10 0.7926, r20 0.7636
q05 배송 중 파손 -> r10 0.7609, r06 0.7061, r02 0.7015
q06 sunscreen that stings the eyes -> r03* 0.7413, r01 0.6091, r09 0.4308
q07 moisturizer for dry skin -> r05* 0.6638, r07 0.4335, r09 0.4102
q08 long-lasting lip tint -> r13* 0.7434, r15 0.6886, r17 0.4991
q09 toner that cleared my acne -> r19 0.7197, r17* 0.7169, r15 0.4907
q10 foundation that oxidizes -> r23* 0.3412, r09 0.2768, r01 0.2328

hits, Korean queries:  3 of 10
hits, English queries: 5 of 10
hits in the other language: 0
all hits: 8 of 20


## 8. Korean texts all look alike to this model

If Korean comes out as single letters, every Korean text gets a similar vector. The cell scores two unrelated
Korean reviews (r04 eyes stinging, r22 broken bottle) and the two English reviews that say the same things (r03, r21).

**Look for:** about 0.684 for the Korean pair and about 0.386 for the English pair. The unrelated Korean pair
scores much higher: a high score between Korean texts means nothing for this model.

In [8]:
idx = {rid: i for i, rid in enumerate(review_ids)}
pair = lambda a, b: float(review_vecs[idx[a]] @ review_vecs[idx[b]])
print("r04 ~ r22 (Korean, unrelated): ", round(pair("r04", "r22"), 4))
print("r03 ~ r21 (English, unrelated):", round(pair("r03", "r21"), 4))

r04 ~ r22 (Korean, unrelated):  0.6841
r03 ~ r21 (English, unrelated): 0.3856


## 9. Mean pooling by hand

The model makes one vector per token. `output_value="token_embeddings"` asks for those token vectors before pooling.
The cell averages them (mean pooling) and compares the result with the vector `encode` returns.
In the lesson you did this with the multilingual model's numbers; here it is the English model, so the numbers differ.

**Look for:** 4 token vectors of 384 numbers for "dry lips", one mean vector of 384 numbers, and a cosine of 1.0
between your mean and the library's sentence vector: they are the same vector.

In [9]:
token_vecs = model.encode("dry lips", output_value="token_embeddings").numpy()
pieces = tok.convert_ids_to_tokens(tok("dry lips")["input_ids"])
print("token vectors:", token_vecs.shape)
for p, v in zip(pieces, token_vecs):
    print(f"  {p:8s}", v[:3].round(4))
mean_vec = token_vecs.mean(axis=0)
print("mean    ", mean_vec[:3].round(4), " shape:", mean_vec.shape)
lib_vec = model.encode("dry lips")
print("cosine(your mean, library vector):", round(float(mean_vec @ lib_vec / (np.linalg.norm(mean_vec) * np.linalg.norm(lib_vec))), 4))

token vectors: (4, 384)
  [CLS]    [-0.5104 -0.2784  0.3389]
  dry      [-0.39   -0.2001  0.1811]
  lips     [-1.318  -0.6562 -0.2955]
  [SEP]    [-1.155  -0.2113  0.3382]
mean     [-0.8433 -0.3365  0.1407]  shape: (384,)
cosine(your mean, library vector): 1.0


## 10. The scoring step from the lesson

This is the `match` function from the lesson's stepped code screen: mean pooling for each text, then cosine
similarity. It starts from token vectors, so it does not use the library's pooling at all.

**Look for:** for "sunscreen that stings the eyes", 0.7413 for the English review r03 (the same score as in step 7)
and only about 0.01 for its Korean twin r04. The lesson's multilingual model scored r04 at 0.6528.

In [10]:
def match(query_tokens, review_tokens):
    query_vec = query_tokens.mean(axis=0)
    review_vec = review_tokens.mean(axis=0)
    dot = query_vec @ review_vec
    query_len = np.linalg.norm(query_vec)
    review_len = np.linalg.norm(review_vec)
    score = dot / (query_len * review_len)
    return score

tokens_of = lambda text: model.encode(text, output_value="token_embeddings").numpy()
q = tokens_of("sunscreen that stings the eyes")
print("query tokens:", q.shape)
for rid in ["r03", "r04"]:
    r = tokens_of(reviews[idx[rid]])
    print(rid, "tokens:", r.shape, " score:", round(float(match(q, r)), 4))

query tokens: (9, 384)
r03 tokens: (20, 384)  score: 0.7413


r04 tokens: (34, 384)  score: 0.0104


## 11. The multilingual model's results, from the lesson's run

The lesson compares the English model with `paraphrase-multilingual-MiniLM-L12-v2`. That model is about 470 MB, so
this notebook does **not** download it. The numbers in the next cell are **copied from the lesson's run files**
(`search.out.txt`, a NumPy forward pass of the multilingual model with its own weights). They are data, not output
of this notebook. The cell counts them with the same `hits` function as step 7.

**Look for:** 6 hits on Korean queries, 5 on English queries, 11 of 20 in all, and 4 hits in the other language,
against 3, 5, 8 and 0 for the English model. The multilingual tokenizer cuts the Korean query into 7 real word pieces.

In [11]:
# Recorded data from the lesson's run (not computed here): top 3 per query, multilingual model.
TOP3_ML = {
    "q01": [("r06", 0.6306), ("r02", 0.6273), ("r10", 0.617)],  "q02": [("r06", 0.7178), ("r16", 0.5965), ("r08", 0.5917)],
    "q03": [("r16", 0.5584), ("r06", 0.4688), ("r15", 0.4215)], "q04": [("r06", 0.3629), ("r10", 0.3526), ("r16", 0.351)],
    "q05": [("r22", 0.6432), ("r10", 0.4565), ("r21", 0.423)],  "q06": [("r03", 0.7622), ("r01", 0.6666), ("r04", 0.6528)],
    "q07": [("r05", 0.7106), ("r06", 0.6902), ("r09", 0.5506)], "q08": [("r15", 0.7101), ("r16", 0.6146), ("r06", 0.5316)],
    "q09": [("r19", 0.5898), ("r15", 0.5157), ("r06", 0.5061)], "q10": [("r23", 0.3278), ("r10", 0.2633), ("r21", 0.1896)],
}
TOKENS_Q01_ML = ["<s>", "▁백", "탁", "▁없는", "▁선", "크림", "</s>"]

print("multilingual tokenizer, q01:", len(TOKENS_Q01_ML), "pieces:", TOKENS_Q01_ML)
print("q07 moisturizer for dry skin -> multilingual:", TOP3_ML["q07"], " English:", TOP3_EN["q07"])
print()
print(f"{'':28s}{'English (run here)':>20s}{'multilingual (recorded)':>26s}")
for label, kw in [("hits, Korean queries", dict(query_lang="ko")), ("hits, English queries", dict(query_lang="en")),
                  ("hits in the other language", dict(other_language_only=True)), ("all hits", {})]:
    print(f"{label:28s}{hits(TOP3_EN, **kw):>20d}{hits(TOP3_ML, **kw):>26d}")

multilingual tokenizer, q01: 7 pieces: ['<s>', '▁백', '탁', '▁없는', '▁선', '크림', '</s>']
q07 moisturizer for dry skin -> multilingual: [('r05', 0.7106), ('r06', 0.6902), ('r09', 0.5506)]  English: [('r05', 0.6638), ('r07', 0.4335), ('r09', 0.4102)]

                              English (run here)   multilingual (recorded)
hits, Korean queries                           3                         6
hits, English queries                          5                         5
hits in the other language                     0                         4
all hits                                       8                        11


If you want to run the multilingual model yourself later, it works on Colab: in step 3 set
`name = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"`, remove the `revision=...` argument and run the
cells again. It downloads about 470 MB. Its scores through the library may differ slightly from the recorded ones in
the 4th decimal; this notebook did not check that.

## Now change it

Each cell below has one line marked `# <- edit this line`. Change it, run the cell, and compare.

### Experiment 1: your own query

The cell searches the 24 reviews with any text you type. As given, "sunscreen that burns my eyes" shares almost no
words with r03 ("ran into my eyes ... stung") and still finds it first, with r01 second. Try a Korean query, for
example `"눈이 따가운 선크림"` (sunscreen that stings the eyes): the English model puts the Korean review r04 first,
because both share the letters of 선크림 and 따가, misses the English review r03, and fills the other two places
with reviews about acne and a cushion (r18, r24). A spelling match, not understanding.

In [12]:
my_query = "sunscreen that burns my eyes"  # <- edit this line
s = model.similarity(model.encode_query([my_query], normalize_embeddings=True), review_vecs)[0].topk(3)
for v, i in zip(s.values, s.indices):
    print(review_ids[i], round(float(v), 4), reviews[i])

r03 0.6958 The sunscreen ran into my eyes when I sweated and they stung for an hour.
r01 0.6277 This sunscreen goes on clear and leaves no chalky film on my face.
r07 0.4062 Way too heavy and greasy for oily skin, my face shines by noon.


### Experiment 2: a long review gets cut

This review is 156 tokens long and its complaint (sweat carried the sunscreen into the eyes) is the last sentence.
As given, the model reads 256 tokens, so it reads the whole review: about 0.630 with the complaint last and 0.604
with it first. Change `READ` to 32. Now the model reads only the first 32 tokens: with the complaint last it is cut
off and the score falls to about 0.404, while with the complaint first it stays in and the score is about 0.720.

In [13]:
story = ("I bought this sunscreen for a week of hiking in the mountains with my sister. The tube is small enough "
         "for a jacket pocket and the cap clicks shut, so nothing leaked in my bag. It smells faintly of citrus, "
         "which I liked. The texture is light and it spreads easily over the arms and the back of the neck. "
         "I put it on every morning before breakfast and again at lunch, and I did not get sunburned once, even on "
         "the long ridge walk on the third day. The price was fair for the size and it arrived two days after I "
         "ordered it. My sister liked it too and plans to buy her own tube before the summer. ")
complaint = "But on the hot afternoons the sweat carried it into my eyes and they stung badly for an hour."
READ = 256  # <- edit this line
model.max_seq_length = READ
qv = model.encode("sunscreen that stings the eyes")
print("tokens in the review:", len(tok(story + complaint)["input_ids"]), " tokens read:", model.max_seq_length)
print("complaint last: ", round(float(model.similarity(qv, model.encode(story + complaint))[0, 0]), 4))
print("complaint first:", round(float(model.similarity(qv, model.encode(complaint + " " + story))[0, 0]), 4))
model.max_seq_length = 256  # put it back for the other cells

tokens in the review: 156  tokens read: 256
complaint last:  0.63
complaint first: 0.6037


### Experiment 3: first token instead of mean pooling

Some models use the first token (`[CLS]`) as the sentence vector instead of the mean. This model was trained with
mean pooling. As given, the cell uses `"mean"` and gets the 8 hits of step 7. Change `POOLING` to `"first"`: every
score jumps up (q06's top score goes from 0.7413 to about 0.945) and the count falls to 7. Scores near 1 do not
mean better matches: pool the way the model was trained.

In [14]:
POOLING = "mean"  # <- edit this line

def sentence_vec(text):
    t = tokens_of(text)
    return t.mean(axis=0) if POOLING == "mean" else t[0]

cosine = lambda a, b: float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))
rvecs = [sentence_vec(text) for text in reviews]
top3 = {}
for qid, _, _, text in QUERIES:
    qv = sentence_vec(text)
    top3[qid] = sorted(((rid, round(cosine(qv, v), 4)) for rid, v in zip(review_ids, rvecs)), key=lambda x: -x[1])[:3]
print("pooling:", POOLING)
print("q06 top 3:", top3["q06"])
print("all hits:", hits(top3), "of 20")

pooling: mean
q06 top 3: [('r03', 0.7413), ('r01', 0.6091), ('r09', 0.4308)]
all hits: 8 of 20


## What you should see

- The English model's tokenizer cuts "백탁 없는 선크림" into 17 pieces, mostly single Korean letters and one `[UNK]`
  (step 4). The recorded multilingual tokenizer gives 7 (step 11).
- Every text becomes 384 numbers however long it is: the 24 reviews give shape `(24, 384)` (step 5).
- The Korean query q01 returns `['r04', 'r12', 'r20']` with `0.8635, 0.7196, 0.6608`, all wrong (step 6).
- Counting hits over the 10 queries gives 8 of 20 for the English model, none in the other language (step 7),
  against 11 of 20 and 4 across languages in the multilingual model's recorded results (step 11).
- Your own mean of the token vectors of "dry lips" has a cosine of 1.0 with the library's sentence vector (step 9).